# Step 1: Single-Race Prototype

**Goal**: For one race (2023 Japan GP), build one feature row per driver using only the first half of laps, attach podium labels, and verify the output.

Reference: `misc/plan_podium_prediction.md`

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os

/Users/chethanjujjavarapu/Desktop/GitHub/kyrios/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load Session

In [2]:
# Cache setup
cache_dir = os.path.join(os.getcwd(), 'cache')
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# Load 2023 Japan GP Race
session = fastf1.get_session(2023, 'Japan', 'R')
session.load()

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '2

## 2. Define Half-Race Point

In [3]:
# Total race laps = max lap completed by the winner
results = session.results
laps = session.laps

total_race_laps = laps['LapNumber'].max()
mid_race_lap = int(np.floor(total_race_laps / 2))

print(f"Total race laps: {total_race_laps}")
print(f"Mid-race lap:    {mid_race_lap}")
print(f"Features use laps 1–{mid_race_lap} only.")

Total race laps: 53.0
Mid-race lap:    26
Features use laps 1–26 only.


## 3. Build Label (Podium = Top 3)

From `session.results`, get each driver's final position. Label = 1 if top 3, else 0.

In [4]:
# Build label dataframe from results
label_df = results[['DriverNumber', 'Abbreviation', 'TeamName', 'GridPosition', 'Position']].copy()
label_df['Position'] = pd.to_numeric(label_df['Position'], errors='coerce')
label_df['GridPosition'] = pd.to_numeric(label_df['GridPosition'], errors='coerce')
label_df['podium'] = (label_df['Position'] <= 3).astype(int)

print(f"Drivers: {len(label_df)}")
print(f"Podium finishers: {label_df['podium'].sum()}")
print()
print(label_df[['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'podium']].to_string(index=False))

Drivers: 20
Podium finishers: 3

Abbreviation        TeamName  GridPosition  Position  podium
         VER Red Bull Racing           1.0       1.0       1
         NOR         McLaren           3.0       2.0       1
         PIA         McLaren           2.0       3.0       1
         LEC         Ferrari           4.0       4.0       0
         HAM        Mercedes           7.0       5.0       0
         SAI         Ferrari           6.0       6.0       0
         RUS        Mercedes           8.0       7.0       0
         ALO    Aston Martin          10.0       8.0       0
         OCO          Alpine          14.0       9.0       0
         GAS          Alpine          12.0      10.0       0
         LAW      AlphaTauri          11.0      11.0       0
         TSU      AlphaTauri           9.0      12.0       0
         ZHO      Alfa Romeo          19.0      13.0       0
         HUL    Haas F1 Team          18.0      14.0       0
         MAG    Haas F1 Team          15.0      15.0

## 4. First-Half Laps Only

Filter laps to only those with `LapNumber <= mid_race_lap`.

In [16]:
mask = laps["LapNumber"] <= mid_race_lap
first_half_laps = laps[mask]

print(f"Total laps in dataset:      {len(laps)}")
print(f"First-half laps (1–{mid_race_lap}):  {len(first_half_laps)}")
print(f"Drivers in first half:      {first_half_laps['DriverNumber'].nunique()}")
first_half_laps.head()

Total laps in dataset:      880
First-half laps (1–26):  480
Drivers in first half:      20


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:05:04.229000,VER,1,0 days 00:02:00.179000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:49.276000,...,True,Red Bull Racing,0 days 01:03:03.844000,2023-09-24 05:04:05.253,124,1.0,False,,False,False
1,0 days 01:07:48.591000,VER,1,NaT,2.0,1.0,NaT,NaT,0 days 00:01:05.251000,0 days 00:01:08.600000,...,True,Red Bull Racing,0 days 01:05:04.229000,2023-09-24 05:06:05.638,4,1.0,False,,False,False
2,0 days 01:10:33.919000,VER,1,NaT,3.0,1.0,NaT,NaT,0 days 00:01:08.931000,0 days 00:01:08.363000,...,True,Red Bull Racing,0 days 01:07:48.591000,2023-09-24 05:08:50.000,4,1.0,False,,False,False
3,0 days 01:13:29.577000,VER,1,NaT,4.0,1.0,NaT,NaT,0 days 00:01:00.867000,0 days 00:01:06.499000,...,True,Red Bull Racing,0 days 01:10:33.919000,2023-09-24 05:11:35.328,41,1.0,False,,False,False
4,0 days 01:15:06.325000,VER,1,0 days 00:01:36.748000,5.0,1.0,NaT,NaT,0 days 00:00:34.900000,0 days 00:00:42.960000,...,True,Red Bull Racing,0 days 01:13:29.577000,2023-09-24 05:14:30.986,1,1.0,False,,False,True


## 5. Build Features Per Driver

### Group 1: Position & Gaps at mid_race_lap

In [ ]:
first_half

In [8]:
# Get each driver's state at the mid_race_lap
# Take the row for each driver at exactly mid_race_lap (or their last lap if they DNF'd before)
mid_lap_state = (
    first_half_laps
    .sort_values('LapNumber')
    .groupby('DriverNumber')
    .last()
    .reset_index()
)

# Position at half-race
position_features = mid_lap_state[['DriverNumber', 'LapNumber', 'Position']].copy()
position_features = position_features.rename(columns={
    'Position': 'position_at_half',
    'LapNumber': 'last_lap_completed'
})

print(position_features.sort_values('position_at_half').to_string(index=False))

DriverNumber  last_lap_completed  position_at_half
           1                26.0               1.0
          81                26.0               2.0
           4                26.0               3.0
          16                26.0               4.0
          55                26.0               5.0
          44                26.0               6.0
          63                26.0               7.0
          31                26.0               8.0
          10                26.0               9.0
          22                26.0              10.0
          24                26.0              11.0
          14                26.0              12.0
          18                20.0              12.0
          27                26.0              13.0
          20                26.0              14.0
          40                26.0              15.0
          23                26.0              16.0
           2                22.0              17.0
          11                15.

### Group 2: Pit Stops & Tyres

In [9]:
# Number of pit stops in first half (count laps where PitInTime is not NaT)
pit_counts = (
    first_half_laps[first_half_laps['PitInTime'].notna()]
    .groupby('DriverNumber')
    .size()
    .reset_index(name='num_pit_stops')
)

# Current tyre state at mid_race_lap (from the last lap in first half)
tyre_state = mid_lap_state[['DriverNumber', 'Compound', 'TyreLife', 'FreshTyre']].copy()
tyre_state = tyre_state.rename(columns={
    'Compound': 'current_compound',
    'TyreLife': 'tyre_age',
    'FreshTyre': 'fresh_tyre'
})

# Merge pit counts (drivers with 0 pit stops won't appear, so fill with 0)
tyre_features = tyre_state.merge(pit_counts, on='DriverNumber', how='left')
tyre_features['num_pit_stops'] = tyre_features['num_pit_stops'].fillna(0).astype(int)

print(tyre_features.to_string(index=False))

DriverNumber current_compound  tyre_age  fresh_tyre  num_pit_stops
           1           MEDIUM      10.0        True              1
          10             HARD       8.0        True              1
          11             SOFT       7.0       False              5
          14             HARD       1.0        True              2
          16           MEDIUM       9.0        True              1
          18           MEDIUM       6.0        True              2
           2             HARD       1.0        True              4
          20             HARD      14.0        True              1
          22           MEDIUM      17.0        True              1
          23             SOFT      13.0        True              3
          24             SOFT      16.0        True              2
          27             HARD       5.0        True              2
          31             HARD      25.0        True              1
           4             HARD       9.0        True           

### Group 3: Pace in First Half

Compute average, best, and std of lap times. Exclude pit in/out laps.

In [10]:
# Filter to "clean" laps: exclude pit in-laps and out-laps
clean_laps = first_half_laps[
    (first_half_laps['PitInTime'].isna()) &   # not a pit in-lap
    (first_half_laps['PitOutTime'].isna()) &   # not a pit out-lap
    (first_half_laps['LapTime'].notna())        # has a valid lap time
].copy()

# Convert LapTime to seconds for numeric operations
clean_laps['LapTime_s'] = clean_laps['LapTime'].dt.total_seconds()

pace_features = (
    clean_laps
    .groupby('DriverNumber')
    .agg(
        avg_lap_time_s=('LapTime_s', 'mean'),
        best_lap_time_s=('LapTime_s', 'min'),
        lap_time_std_s=('LapTime_s', 'std'),
        clean_laps_count=('LapTime_s', 'count')
    )
    .reset_index()
)

# Also get total laps completed in first half (including pit laps)
laps_completed = (
    first_half_laps
    .groupby('DriverNumber')
    .size()
    .reset_index(name='laps_completed')
)

pace_features = pace_features.merge(laps_completed, on='DriverNumber', how='left')

print(pace_features.round(3).to_string(index=False))

DriverNumber  avg_lap_time_s  best_lap_time_s  lap_time_std_s  clean_laps_count  laps_completed
           1         100.020           96.748           6.648                21              26
          10         102.604           97.279           9.840                21              26
          11         104.189           99.704          10.770                 8              15
          14         102.128           98.126           7.256                19              26
          16         101.087           97.880           7.353                21              26
          18         102.798           99.050          10.572                14              20
           2         102.093           98.848           5.161                14              22
          20         101.988           98.373           9.530                21              26
          22         102.902           98.737           8.599                21              26
          23         100.486           9

### Group 4: Track Status & Weather (First Half Only)

In [11]:
# --- Track status in first half ---
# Get the time at the end of mid_race_lap (use the max Time from first_half_laps)
half_race_time = first_half_laps['Time'].max()

track_status = session.track_status
ts_first_half = track_status[track_status['Time'] <= half_race_time].copy()

sc_deployed = int((ts_first_half['Status'] == '4').any())
vsc_deployed = int((ts_first_half['Status'] == '6').any())

# Count yellow/SC/VSC status changes in first half
yellow_count = int((ts_first_half['Status'] == '2').sum())

print(f"Safety Car deployed in first half:  {bool(sc_deployed)}")
print(f"VSC deployed in first half:         {bool(vsc_deployed)}")
print(f"Yellow flag changes in first half:  {yellow_count}")

# --- Weather in first half ---
weather = session.weather_data
weather_first_half = weather[weather['Time'] <= half_race_time].copy()

avg_air_temp = weather_first_half['AirTemp'].mean()
avg_track_temp = weather_first_half['TrackTemp'].mean()
rain_in_first_half = int(weather_first_half['Rainfall'].any())

print(f"\nAvg air temp (first half):    {avg_air_temp:.1f}°C")
print(f"Avg track temp (first half):  {avg_track_temp:.1f}°C")
print(f"Rain in first half:           {bool(rain_in_first_half)}")

# These are race-level (same for all drivers), so we store them as scalars
race_conditions = {
    'sc_deployed': sc_deployed,
    'vsc_deployed': vsc_deployed,
    'yellow_count': yellow_count,
    'rain_in_first_half': rain_in_first_half,
    'avg_air_temp': round(avg_air_temp, 1),
    'avg_track_temp': round(avg_track_temp, 1),
}

Safety Car deployed in first half:  True
VSC deployed in first half:         True
Yellow flag changes in first half:  2

Avg air temp (first half):    27.3°C
Avg track temp (first half):  42.9°C
Rain in first half:           False


### Group 5: Pre-Race Context

In [12]:
# Pre-race context from results and event
pre_race = label_df[['DriverNumber', 'Abbreviation', 'TeamName', 'GridPosition']].copy()
pre_race['round_number'] = session.event.RoundNumber
pre_race['track'] = session.event.Location
pre_race['country'] = session.event.Country
pre_race['total_race_laps'] = total_race_laps
pre_race['mid_race_lap'] = mid_race_lap
pre_race['laps_remaining'] = total_race_laps - mid_race_lap

print(pre_race.to_string(index=False))

DriverNumber Abbreviation        TeamName  GridPosition  round_number  track country  total_race_laps  mid_race_lap  laps_remaining
           1          VER Red Bull Racing           1.0            16 Suzuka   Japan               53            26              27
           4          NOR         McLaren           3.0            16 Suzuka   Japan               53            26              27
          81          PIA         McLaren           2.0            16 Suzuka   Japan               53            26              27
          16          LEC         Ferrari           4.0            16 Suzuka   Japan               53            26              27
          44          HAM        Mercedes           7.0            16 Suzuka   Japan               53            26              27
          55          SAI         Ferrari           6.0            16 Suzuka   Japan               53            26              27
          63          RUS        Mercedes           8.0            16 Suzuka

## 6. Assemble Final Feature Table

Merge all feature groups + label into one row per driver.

In [13]:
# Start with pre-race context
df = pre_race.copy()

# Merge position features
df = df.merge(position_features, on='DriverNumber', how='left')

# Derive positions_gained (positive = gained places from grid)
df['positions_gained'] = df['GridPosition'] - df['position_at_half']

# Merge tyre features
df = df.merge(tyre_features, on='DriverNumber', how='left')

# Merge pace features
df = df.merge(pace_features, on='DriverNumber', how='left')

# Add race-level conditions (same for all drivers)
for col, val in race_conditions.items():
    df[col] = val

# Add label
df = df.merge(label_df[['DriverNumber', 'Position', 'podium']], on='DriverNumber', how='left')
df = df.rename(columns={'Position': 'final_position'})

# Add a flag for drivers who DNF'd before mid_race_lap
df['still_running_at_half'] = (df['last_lap_completed'] >= mid_race_lap).astype(int)

# Sort by position at half-race
df = df.sort_values('position_at_half').reset_index(drop=True)

print(f"Rows: {len(df)}")
print(f"Podium labels (1): {df['podium'].sum()}")
print(f"Non-podium labels (0): {(df['podium'] == 0).sum()}")
print()
df

Rows: 20
Podium labels (1): 3
Non-podium labels (0): 17



,DriverNumber,Abbreviation,TeamName,GridPosition,round_number,track,country,total_race_laps,mid_race_lap,laps_remaining,...,laps_completed,sc_deployed,vsc_deployed,yellow_count,rain_in_first_half,avg_air_temp,avg_track_temp,final_position,podium,still_running_at_half
0,1,VER,Red Bull Racing,1.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,1.0,1,1
1,81,PIA,McLaren,2.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,3.0,1,1
2,4,NOR,McLaren,3.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,2.0,1,1
3,16,LEC,Ferrari,4.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,4.0,0,1
4,55,SAI,Ferrari,6.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,6.0,0,1
5,44,HAM,Mercedes,7.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,5.0,0,1
6,63,RUS,Mercedes,8.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,7.0,0,1
7,31,OCO,Alpine,14.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,9.0,0,1
8,10,GAS,Alpine,12.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,10.0,0,1
9,22,TSU,AlphaTauri,9.0,16,Suzuka,Japan,53,26,27,...,26,1,1,2,0,27.3,42.9,12.0,0,1


## 7. Verification

Check that:
1. We have ~20 rows (one per driver).
2. Exactly 3 rows have `podium = 1`.
3. No feature uses data beyond `mid_race_lap`.

In [14]:
# Verification checks
checks_passed = True

# Check 1: Row count
n_drivers = len(df)
print(f"[CHECK 1] Rows (drivers): {n_drivers}")
if n_drivers < 15 or n_drivers > 22:
    print("  WARNING: unexpected number of drivers")
    checks_passed = False
else:
    print("  PASS")

# Check 2: Exactly 3 podium labels
n_podium = df['podium'].sum()
print(f"\n[CHECK 2] Podium labels: {n_podium}")
if n_podium != 3:
    print(f"  WARNING: expected 3 podium finishers, got {n_podium}")
    checks_passed = False
else:
    print("  PASS")

# Check 3: No laps beyond mid_race_lap used in features
max_lap_used = first_half_laps['LapNumber'].max()
print(f"\n[CHECK 3] Max lap used in features: {max_lap_used} (mid_race_lap = {mid_race_lap})")
if max_lap_used > mid_race_lap:
    print("  FAIL: features use laps beyond mid_race_lap!")
    checks_passed = False
else:
    print("  PASS")

# Check 4: Podium drivers match actual top 3
actual_podium = df[df['podium'] == 1]['Abbreviation'].tolist()
print(f"\n[CHECK 4] Actual podium: {actual_podium}")
print(f"  Final positions of podium drivers: {df[df['podium'] == 1]['final_position'].tolist()}")
print("  PASS" if all(p <= 3 for p in df[df['podium'] == 1]['final_position']) else "  FAIL")

# Summary
print(f"\n{'='*40}")
print(f"ALL CHECKS PASSED: {checks_passed}")

[CHECK 1] Rows (drivers): 20
  PASS

[CHECK 2] Podium labels: 3
  PASS

[CHECK 3] Max lap used in features: 26.0 (mid_race_lap = 26)
  PASS

[CHECK 4] Actual podium: ['VER', 'PIA', 'NOR']
  Final positions of podium drivers: [1.0, 3.0, 2.0]
  PASS

ALL CHECKS PASSED: True


## 8. Preview: Feature Columns

List all columns that would be used as model features (excluding identifiers and label).

In [15]:
# Columns that are identifiers or labels (not features)
non_feature_cols = [
    'DriverNumber', 'Abbreviation', 'final_position', 'podium',
]

# Columns that need encoding before modeling (categorical)
categorical_cols = ['TeamName', 'current_compound', 'track', 'country']

# All other columns are numeric features
feature_cols = [c for c in df.columns if c not in non_feature_cols]

print("Feature columns:")
for col in feature_cols:
    dtype = 'categorical' if col in categorical_cols else 'numeric'
    print(f"  {col:30s} ({dtype})")

print(f"\nTotal features: {len(feature_cols)}")
print(f"  Numeric:     {len(feature_cols) - len(categorical_cols)}")
print(f"  Categorical: {len(categorical_cols)}")

Feature columns:
  TeamName                       (categorical)
  GridPosition                   (numeric)
  round_number                   (numeric)
  track                          (categorical)
  country                        (categorical)
  total_race_laps                (numeric)
  mid_race_lap                   (numeric)
  laps_remaining                 (numeric)
  last_lap_completed             (numeric)
  position_at_half               (numeric)
  positions_gained               (numeric)
  current_compound               (categorical)
  tyre_age                       (numeric)
  fresh_tyre                     (numeric)
  num_pit_stops                  (numeric)
  avg_lap_time_s                 (numeric)
  best_lap_time_s                (numeric)
  lap_time_std_s                 (numeric)
  clean_laps_count               (numeric)
  laps_completed                 (numeric)
  sc_deployed                    (numeric)
  vsc_deployed                   (numeric)
  yellow_count       

## 9. Summary

**What we built:**
- One row per driver for the 2023 Japan GP.
- Features computed only from laps 1 through `mid_race_lap` and pre-race info.
- Label = 1 for top-3 finishers, 0 for everyone else.

**Next step (Step 2):**
- Scale this to multiple races across multiple seasons.
- Stack all rows into one big DataFrame.
- Add race identifiers (year, round) for time-based train/val/test splitting.